# QQQI / QQQ / TQQQ current strategy review

This is the rolling comparison notebook for the frozen v4.1 baseline and the v4.2 50/50 bridge challenger.

- Baseline: `qqqi_qqq_tqqq_vxn_leverage_v4_1`
- Challenger: `qqqi_qqq_tqqq_vxn_bridge_v4_2`
- Prospective boundary: 2026-08-01
- Status: research-only; neither candidate is trade-ready.

The notebook is refreshed with `scripts/refresh_qqqi_vxn_current_notebook.py`. It displays the durable published snapshot first, then recomputes the current live-data comparison using the frozen contracts.

In [ ]:
from pathlib import Path
import json
import os
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from IPython.display import display, Markdown

from src.research.etf_rotation_experiment import fetch_adjusted_daily_bars
from src.research.vxn_bridge_allocation_experiment import run_bridge_allocation_comparison
from src.research.vxn_prospective_monitor import prospective_return_metrics

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
SNAPSHOT_PATH = ROOT / 'docs/research/snapshots/qqqi_vxn_v4_1_v4_2_2026-07-31.json'
CONTRACT_PATH = ROOT / 'configs/research_paradigms/qqqi_qqq_tqqq_vxn_bridge_v4_2.yaml'
END_DATE = os.getenv('QQQI_VXN_NOTEBOOK_END_DATE')
if END_DATE is None:
    END_DATE = (pd.Timestamp.utcnow().normalize() + pd.Timedelta(days=1)).date().isoformat()
BASELINE = 'rotation_vxn_leverage_v4_1_75'
BRIDGE = 'rotation_vxn_bridge_v4_2_50_50'
PROSPECTIVE_START = '2026-08-01'
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)
print(f'Repository root: {ROOT}')
print(f'Live-data end boundary (exclusive): {END_DATE}')


## 1. Durable published snapshot

The JSON snapshot is the compact, permanent record of published metrics and evidence identifiers. Full daily traces remain in the referenced GitHub Actions artifacts.

In [ ]:
snapshot = json.loads(SNAPSHOT_PATH.read_text(encoding='utf-8'))
published_rows = []
for key, values in snapshot['strategies'].items():
    if 'cagr' not in values:
        continue
    published_rows.append({
        'strategy': key,
        'role': values.get('role', 'benchmark'),
        'total_return': values.get('total_return'),
        'cagr': values.get('cagr'),
        'annual_volatility': values.get('annual_volatility'),
        'sharpe': values.get('sharpe'),
        'sortino': values.get('sortino'),
        'max_drawdown': values.get('max_drawdown'),
        'calmar': values.get('calmar'),
        'turnover_units': values.get('turnover_units'),
    })
published = pd.DataFrame(published_rows).set_index('strategy')
display(published.style.format({
    'total_return': '{:.2%}',
    'cagr': '{:.2%}',
    'annual_volatility': '{:.2%}',
    'sharpe': '{:.3f}',
    'sortino': '{:.3f}',
    'max_drawdown': '{:.2%}',
    'calmar': '{:.3f}',
    'turnover_units': '{:.1f}',
}))
display(pd.DataFrame(snapshot['evidence']).T)


## 2. Recompute the frozen live-data comparison

Both candidates use the exact same state decisions. The only difference is state 1 allocation: v4.1 holds 100% QQQ, while v4.2 holds 50% QQQI and 50% QQQ.

In [ ]:
contract = yaml.safe_load(CONTRACT_PATH.read_text(encoding='utf-8'))
boundaries = contract['boundaries']
symbols = [*boundaries['tradable_symbols'], boundaries['vix_symbol'], boundaries['vxn_symbol']]
bars, coverage = fetch_adjusted_daily_bars(
    symbols=list(dict.fromkeys(symbols)),
    start=contract['data']['start_date'],
    end=END_DATE,
)
metrics, results, prepared, diagnostics = run_bridge_allocation_comparison(bars, contract)
display(coverage)
metric_columns = [
    'total_return', 'cagr', 'annual_volatility', 'sharpe', 'sortino',
    'max_drawdown', 'calmar', 'switch_count', 'turnover_units',
    'transaction_cost_paid', 'pct_time_qqqi', 'pct_time_qqq',
    'pct_time_partial_tqqq', 'average_tqqq_weight',
]
display(metrics.loc[[key for key in ('buy_hold_QQQ', BASELINE, BRIDGE) if key in metrics.index], metric_columns])
print(f'Latest aligned feature date: {prepared.index.max().date()}')
print(f'Latest economic return date: {results[BRIDGE].daily.index.max().date()}')


In [ ]:
names = {
    'buy_hold_QQQ': 'QQQ buy and hold',
    BASELINE: 'v4.1 baseline',
    BRIDGE: 'v4.2 50/50 bridge',
}
equity = pd.concat({names[key]: results[key].daily['equity'] for key in names}, axis=1)
ax = equity.plot(figsize=(14, 5), title='Current full-portfolio equity')
ax.set_ylabel('Equity')
ax.grid(True, alpha=0.3)
plt.show()
drawdown = equity.div(equity.cummax()).sub(1.0)
ax = drawdown.plot(figsize=(14, 4), title='Current full-portfolio drawdown')
ax.set_ylabel('Drawdown')
ax.grid(True, alpha=0.3)
plt.show()


## 3. State equality and allocation attribution

A valid bridge comparison must preserve every v4.1 state date. Any state divergence is a research-contract failure.

In [ ]:
baseline_daily = results[BASELINE].daily.copy()
bridge_daily = results[BRIDGE].daily.copy()
common = baseline_daily.index.intersection(bridge_daily.index)
state_equal = baseline_daily.loc[common, 'position_state'].equals(bridge_daily.loc[common, 'position_state'])
assert state_equal, 'v4.1 and v4.2 state traces diverged'
allocation_summary = pd.DataFrame({
    'v4.1': ['100% QQQI', '100% QQQ', '25% QQQ + 75% TQQQ'],
    'v4.2 bridge': ['100% QQQI', '50% QQQI + 50% QQQ', '25% QQQ + 75% TQQQ'],
}, index=['state 0: defense', 'state 1: early recovery', 'state 2: confirmed recovery'])
display(allocation_summary)
comparison = pd.DataFrame({
    'v4.1_turnover': [baseline_daily['turnover_units'].sum()],
    'v4.2_turnover': [bridge_daily['turnover_units'].sum()],
    'turnover_saved': [baseline_daily['turnover_units'].sum() - bridge_daily['turnover_units'].sum()],
    'v4.1_cost': [baseline_daily['transaction_cost'].sum()],
    'v4.2_cost': [bridge_daily['transaction_cost'].sum()],
    'state_trace_equal': [state_equal],
})
display(comparison)


## 4. v4.1 signal dates and next-open executed trades

Signals are produced after the close and are executed at the next adjusted open. The markers below must therefore not be interpreted as same-close fills.

In [ ]:
overlay = baseline_daily
state_names = {0: 'QQQI defensive', 1: 'QQQ attack', 2: '25% QQQ + 75% TQQQ'}
action_names = {
    (0, 1): 'Enter QQQ',
    (1, 2): 'Add TQQQ leverage',
    (2, 1): 'Reduce TQQQ leverage',
    (1, 0): 'Move to QQQI defense',
    (2, 0): 'Move to QQQI defense',
}
switches = overlay['position_state'].ne(overlay['position_state'].shift())
execution_rows = overlay.loc[switches].iloc[1:]
records = []
for execution_date, execution_row in execution_rows.iterrows():
    location = overlay.index.get_loc(execution_date)
    prior_row = overlay.iloc[location - 1]
    from_state = int(prior_row['position_state'])
    to_state = int(execution_row['position_state'])
    record = {
        'signal_date': overlay.index[location - 1],
        'execution_date': execution_date,
        'action': action_names.get((from_state, to_state), f'{from_state}->{to_state}'),
        'executed_reason': execution_row['executed_reason'],
        'signal_QQQ_close': prior_row['qqq_close'],
        'signal_VIX': prior_row['vix_close'],
        'signal_VXN': prior_row['vxn_close'],
    }
    orders = []
    for asset in ('QQQI', 'QQQ', 'TQQQ'):
        old_weight = float(prior_row[f'weight_{asset}'])
        new_weight = float(execution_row[f'weight_{asset}'])
        delta = new_weight - old_weight
        record[f'delta_weight_{asset}'] = delta
        if delta > 1e-12:
            orders.append(f'BUY {delta:.0%} {asset}')
        elif delta < -1e-12:
            orders.append(f'SELL {abs(delta):.0%} {asset}')
    record['orders'] = ' | '.join(orders)
    records.append(record)
trade_events = pd.DataFrame(records)
display(trade_events)
markers = {'Enter QQQ': '^', 'Add TQQQ leverage': 'P', 'Reduce TQQQ leverage': 'v', 'Move to QQQI defense': 'X'}
ax = overlay['qqq_close'].plot(figsize=(14, 5), title='QQQ close with close-derived v4.1 signals')
for action, marker in markers.items():
    subset = trade_events[trade_events['action'].eq(action)]
    if not subset.empty:
        ax.scatter(subset['signal_date'], subset['signal_QQQ_close'], marker=marker, s=80, label=action)
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

for asset in ('QQQI', 'QQQ', 'TQQQ'):
    price_column = f'{asset}_open'
    delta_column = f'delta_weight_{asset}'
    buys = trade_events[trade_events[delta_column].gt(1e-12)]
    sells = trade_events[trade_events[delta_column].lt(-1e-12)]
    ax = overlay[price_column].plot(figsize=(14, 4), title=f'{asset} next-open executed trades')
    if not buys.empty:
        ax.scatter(buys['execution_date'], overlay.loc[buys['execution_date'], price_column], marker='^', s=80, label='Buy or increase')
    if not sells.empty:
        ax.scatter(sells['execution_date'], overlay.loc[sells['execution_date'], price_column], marker='v', s=80, label='Sell or reduce')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()


## 5. Prospective observations

Only economic returns dated on or after 2026-08-01 are labelled prospective. Historical recomputation remains context, not new out-of-sample evidence.

In [ ]:
prospective_rows = []
for key in ('buy_hold_QQQ', BASELINE, BRIDGE):
    values = prospective_return_metrics(results[key], PROSPECTIVE_START)
    values['strategy'] = key
    prospective_rows.append(values)
prospective = pd.DataFrame(prospective_rows).set_index('strategy')
display(prospective)
if prospective['observations'].max() == 0:
    display(Markdown('**Status: awaiting the first prospective economic return. No historical data are backfilled.**'))


## Interpretation boundary

- v4.1 remains the frozen baseline.
- v4.2 remains a post-result allocation challenger, not a validated replacement.
- The bridge improves retrospective net metrics mainly through lower transition friction; it does not create a new signal.
- Do not search additional bridge weights, VIX/VXN thresholds, persistence rules or factors on the same sample.
- Reconsider the comparison only after the predeclared prospective evidence threshold is reached.